# Multi-view wall reconstruction (DUSt3R) — Colab

Recovers true 3D wall geometry (incl. folds/panel angles) from several overlapping views of the same wall. Runs on a Colab **GPU**.

**Batch:** upload `wall_sets_batch.zip` (several geometry-verified walls) → reconstructs **every** wall, prints each fold angle, downloads one `out.zip` with per-wall profile + point cloud.
**Single:** upload `wall_0310_set.zip` instead (one wall).

**Steps:** Runtime → Change runtime type → **GPU**, then Run all. Self-contained + re-runnable (reuses already-uploaded images).

In [ ]:
# 1. Install deps + clone DUSt3R (idempotent — safe to re-run)
import os
!pip -q install roma einops trimesh matplotlib opencv-python 'huggingface-hub[torch]>=0.22'
if not os.path.exists('/content/dust3r'):
    !git clone --recursive --depth 1 https://github.com/naver/dust3r.git /content/dust3r
else:
    print('dust3r already cloned')

In [ ]:
# 2. Images. BATCH mode: upload wall_sets_batch.zip (several walls under sets/wall_*/images).
#    Single mode: upload wall_0310_set.zip (one wall under images/). Reuses files if present.
import os, glob, zipfile
have = glob.glob('/content/sets/*/images/*.jpg') or glob.glob('/content/images/*.jpg')
if have:
    print('images already present:', len(have))
else:
    from google.colab import files
    up = files.upload()          # pick wall_sets_batch.zip (batch) OR wall_0310_set.zip (single)
    for name in up:
        if name.lower().endswith('.zip'):
            dest = '/content/sets' if 'batch' in name.lower() else '/content'
            os.makedirs(dest, exist_ok=True)
            with zipfile.ZipFile(name) as z:
                z.extractall(dest)
    print('batch walls:', len(glob.glob('/content/sets/*/images')),
          '| single imgs:', len(glob.glob('/content/images/*.jpg')))

In [ ]:
# 3. Reconstruct — loops over every wall (batch) or the single set. Self-contained.
import os, sys, glob, gc, numpy as np, torch
sys.path.insert(0, '/content/dust3r'); sys.path.insert(0, '/content/dust3r/croco')
from dust3r.inference import inference
from dust3r.model import AsymmetricCroCo3DStereo
from dust3r.utils.image import load_images
from dust3r.image_pairs import make_pairs
from dust3r.cloud_opt import global_aligner, GlobalAlignerMode

device = 'cuda'
WEIGHTS = 'naver/DUSt3R_ViTLarge_BaseDecoder_512_dpt'
print('loading DUSt3R...')
MODEL = AsymmetricCroCo3DStereo.from_pretrained(WEIGHTS).to(device)

def fit_plane(pts, iters=400, thresh=0.02, rng=None):
    rng = rng or np.random.default_rng(0)
    fit = pts if len(pts) <= 120000 else pts[rng.choice(len(pts), 120000, replace=False)]
    best_in, best = None, None
    for _ in range(iters):
        idx = rng.choice(len(fit), 3, replace=False); p = fit[idx]
        n = np.cross(p[1]-p[0], p[2]-p[0]); nn = np.linalg.norm(n)
        if nn < 1e-9: continue
        n = n/nn; d = -n@p[0]; inl = np.abs(fit@n + d) < thresh
        if best_in is None or inl.sum() > best_in.sum(): best_in, best = inl, (n, d)
    n, d = best; c = fit[best_in].mean(0)
    _, _, vh = np.linalg.svd(fit[best_in]-c, full_matrices=False); n = vh[-1]; n /= np.linalg.norm(n)+1e-9
    return n, float(-n@c)

def reconstruct(img_paths, name):
    imgs_in = load_images(img_paths[:8], size=512)
    pairs = make_pairs(imgs_in, scene_graph='complete', prefilter=None, symmetrize=True)
    output = inference(pairs, MODEL, device, batch_size=1)
    scene = global_aligner(output, device=device, mode=GlobalAlignerMode.PointCloudOptimizer)
    scene.compute_global_alignment(init='mst', niter=300, schedule='cosine', lr=0.01)
    with torch.no_grad():
        pts3d = [p.detach().cpu().numpy() for p in scene.get_pts3d()]
        confs = [c.detach().cpu().numpy() for c in scene.get_conf()]
        imgs = [im for im in scene.imgs]
    del scene, output; gc.collect(); torch.cuda.empty_cache()
    pts, cols = [], []
    for p, c, im in zip(pts3d, confs, imgs):
        mk = c > np.percentile(c, 40); pts.append(p[mk]); cols.append((im[mk]*255).astype('uint8'))
    P = np.concatenate(pts); C = np.concatenate(cols)
    n1, d1 = fit_plane(P); in1 = np.abs(P@n1+d1) < 0.02; rest = P[~in1]
    fold = None
    if len(rest) > 2000:
        n2, d2 = fit_plane(rest); in2 = np.abs(rest@n2+d2) < 0.02
        if in2.sum() > 0.1*len(P):
            fold = float(np.degrees(np.arccos(np.clip(abs(n1@n2), 0, 1))))
    od = f'/content/out/{name}'; os.makedirs(od, exist_ok=True)
    import trimesh; trimesh.PointCloud(P, colors=C).export(f'{od}/wall_cloud.ply')
    import matplotlib; matplotlib.use('Agg'); import matplotlib.pyplot as plt
    sub = P[np.random.default_rng(0).choice(len(P), min(40000, len(P)), replace=False)]
    fig, axs = plt.subplots(1, 2, figsize=(14, 6))
    axs[0].scatter(sub[:,0], sub[:,2], s=1, c=sub[:,1], cmap='viridis')
    axs[0].set_title(f'{name} top-down (X-Z) — fold={fold:.1f} deg' if fold else f'{name} top-down (X-Z)'); axs[0].axis('equal')
    axs[1].scatter(sub[:,0], -sub[:,1], s=1, c=sub[:,2], cmap='magma'); axs[1].set_title('front (X-Y)'); axs[1].axis('equal')
    fig.tight_layout(); fig.savefig(f'{od}/wall_profile.png', dpi=90); plt.close(fig)
    print(f'  {name}: {len(P):,} pts | fold = {("%.1f deg" % fold) if fold else "single plane"} -> out/{name}/')
    return fold

# discover walls: batch sets, else single images/
walls = sorted(glob.glob('/content/sets/*'))
jobs = [(os.path.basename(w), sorted(glob.glob(f'{w}/images/*.jpg'))) for w in walls] \
       or [('wall', sorted(glob.glob('/content/images/*.jpg')))]
print(f'{len(jobs)} wall(s) to reconstruct')
for name, paths in jobs:
    if paths: reconstruct(paths, name)
del MODEL; gc.collect(); torch.cuda.empty_cache()
print('done.')

In [ ]:
# 4. Show every wall's profile + zip all outputs for download
import os, glob
from IPython.display import Image as IPImage, display
profiles = sorted(glob.glob('/content/out/*/wall_profile.png'))
if profiles:
    for p in profiles:
        display(IPImage(p))
    !cd /content && zip -qr out.zip out
    from google.colab import files
    files.download('/content/out.zip')   # per-wall wall_profile.png + wall_cloud.ply
else:
    print('No output yet — run cell 3 first.')